In [ ]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import json
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Load the JSON files
try:
    with open('movements1.json', 'r') as file:
        movements1 = json.load(file)
    with open('movements2.json', 'r') as file:
        movements2 = json.load(file)
except FileNotFoundError:
    logging.error("The file 'movements.json' was not found.")
    movements1, movements2 = [], []

# Function to search for matching movements and return the third value
def find_ar(movements, start_point, end_point):
    start_point_rounded = [round(coord, 2) for coord in start_point]
    end_point_rounded = [round(coord, 2) for coord in end_point]

    for movement in movements:
        movement_start_rounded = [round(coord, 2) for coord in movement[0]]
        movement_end_rounded = [round(coord, 2) for coord in movement[1]]
        
        if movement_start_rounded == start_point_rounded and movement_end_rounded == end_point_rounded:
            return movement[2]
    return None

# Check if CUDA is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the environment
GRID_SIZE = 11
ACTIONS = ['up', 'down', 'left', 'right']
ACTION_MAP = {
    'up': (-1, 0),
    'down': (1, 0),
    'left': (0, -1),
    'right': (0, 1)
}

class QNetwork(nn.Module):
    def __init__(self):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(GRID_SIZE ** 2, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, len(ACTIONS))

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

def normalize_coordinates(row, col, grid_size):
    return 0.2 + (1.6 * row / (grid_size - 1)), 0.2 + (1.6 * col / (grid_size - 1))

def train_q_network(q_network, target_network, optimizer, replay_buffer, gamma, batch_size):
    if len(replay_buffer) < batch_size:
        return

    batch = replay_buffer.sample(batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)

    states_tensor = torch.Tensor(states).to(device)
    next_states_tensor = torch.Tensor(next_states).to(device)
    actions_tensor = torch.tensor(actions).long().to(device)
    rewards_tensor = torch.tensor(rewards).to(device)
    dones_tensor = torch.tensor(dones).float().to(device)

    q_values = q_network(states_tensor)
    next_q_values = q_network(next_states_tensor).detach()
    next_q_values_target = target_network(next_states_tensor).detach()

    max_next_actions = torch.argmax(next_q_values, dim=1)
    next_q_targets = rewards_tensor + gamma * next_q_values_target.gather(1, max_next_actions.unsqueeze(1)).squeeze(1) * (1 - dones_tensor)

    optimizer.zero_grad()
    loss = nn.MSELoss()(q_values.gather(1, actions_tensor.unsqueeze(1)), next_q_targets.unsqueeze(1))
    loss.backward()
    optimizer.step()

def deep_q_learning(episodes, min_batch_size=64):
    q_network = QNetwork().to(device)
    target_network = QNetwork().to(device)
    target_network.load_state_dict(q_network.state_dict())
    target_network.eval()

    optimizer = optim.Adam(q_network.parameters(), lr=0.001)
    replay_buffer = ReplayBuffer()
    gamma = 0.99
    epsilon = 1.0
    epsilon_min = 0.01
    epsilon_decay = 0.995
    target_update_interval = 10
    cumulative_rewards = []
    episode_paths = []
    normalized_paths = []

    for episode in range(episodes):
        state = np.zeros((GRID_SIZE, GRID_SIZE))
        state[0, 0] = 1
        visited = set([(0, 0)])
        path = [(0, 0)]
        normalized_path = [(0.2, 0.2)]
        done = False
        total_reward = 0

        while not done:
            if np.random.rand() < epsilon:
                action = np.random.choice(ACTIONS)
            else:
                q_values = q_network(torch.Tensor(state.flatten()).to(device))
                action = ACTIONS[torch.argmax(q_values).item()]

            curr_row, curr_col = np.argwhere(state == 1)[0]
            next_row, next_col = curr_row + ACTION_MAP[action][0], curr_col + ACTION_MAP[action][1]

            if next_row < 0 or next_row >= GRID_SIZE or next_col < 0 or next_col >= GRID_SIZE:
                reward = -1
                done = True
                next_state = state.copy()
            elif (next_row, next_col) in visited:
                reward = -1
                done = True
            else:
                next_state = state.copy()
                next_state[curr_row, curr_col] = 0
                next_state[next_row, next_col] = 1
                visited.add((next_row, next_col))
                path.append((next_row, next_col))
                norm_x1, norm_y1 = normalize_coordinates(curr_row, curr_col, GRID_SIZE)
                norm_x2, norm_y2 = normalize_coordinates(next_row, next_col, GRID_SIZE)
                AR = find_ar(movements1, [norm_x1, norm_y1], [norm_x2, norm_y2])
                VOL = find_ar(movements2, [norm_x1, norm_y1], [norm_x2, norm_y2])
                reward1 = (1 / AR)*10  
                reward2 = (1/(VOL*10**13))*100
                alpha= .5
                beta= .9
                reward =alpha* reward1 + beta* reward2
                normalized_path.append((norm_x2, norm_y2))

            total_reward += reward
            replay_buffer.push(state.flatten(), ACTIONS.index(action), reward, next_state.flatten(), done)
            state = next_state

            train_q_network(q_network, target_network, optimizer, replay_buffer, gamma, min_batch_size)

            epsilon = max(epsilon * epsilon_decay, epsilon_min)

            if episode % target_update_interval == 0:
                target_network.load_state_dict(q_network.state_dict())

        unvisited_penalty = -10 * (GRID_SIZE * GRID_SIZE - len(visited))
        total_reward += unvisited_penalty
        cumulative_rewards.append(total_reward)
        episode_paths.append(path)
        normalized_paths.append(normalized_path)

        if episode % 100 == 0:
            logging.info(f'Episode: {episode},Step: {len(path)}, Total Reward: {total_reward}, reward1:{reward1}, reward2:{reward2}')

    return q_network, cumulative_rewards, episode_paths, normalized_paths

def plot_paths(normalized_paths, grid_size, interval=200):
    for i, norm_path in enumerate(normalized_paths):
        if (i + 1) % interval == 0:
            plt.figure(figsize=(5, 5))
            plt.title(f'Episode {i + 1}')
            plt.grid(True)
            plt.xlim(0.15, 1.85)
            plt.ylim(0.15, 1.85)

            x_coords, y_coords = zip(*norm_path)
            plt.plot(x_coords, y_coords, 'o-', color='blue', markersize=5, linewidth=2)
            plt.xticks(np.linspace(0.2, 1.85, grid_size), [f'{x:.2f}' for x in np.linspace(0.2, 1.85, grid_size)])
            plt.yticks(np.linspace(0.2, 1.85, grid_size), [f'{y:.2f}' for y in np.linspace(0.2, 1.85, grid_size)])
            plt.show()

            save_normalized_path(norm_path, f'normalized_path_episode_{i + 1}.txt')

def save_normalized_path(normalized_path, filename):
    with open(filename, 'w') as f:
        x_coords = [f'{x:.2f}E-3,' for x, _ in normalized_path]
        f.write(' '.join(x_coords) + '\n')
        
        y_coords = [f'{y:.2f}E-3,' for _, y in normalized_path]
        f.write(' '.join(y_coords) + '\n')
        
        f.write(' '.join(['0.1E-3,'] * len(normalized_path)) + '\n')
        
        f.write(' '.join(['1,'] * len(normalized_path)) + '\n')

# Train the agent and get the cumulative rewards and episode paths
q_network, cumulative_rewards, episode_paths, normalized_paths = deep_q_learning(episodes=200000)

# Plotting the cumulative rewards with moving average
plt.figure(figsize=(10, 5))
window = 50
cumulative_rewards_smoothed = np.convolve(cumulative_rewards, np.ones(window)/window, mode='valid')
plt.plot(range(len(cumulative_rewards_smoothed)), cumulative_rewards_smoothed, label='Cumulative Rewards')
plt.title('Average Cumulative Reward per Episode')
plt.xlabel('Episode')
plt.ylabel('Cumulative Reward')
plt.legend()
plt.show()

# Plotting the agent's paths after every 200 episodes
plot_paths(normalized_paths, GRID_SIZE)
